In [ ]:
import glob, hashlib, json, os, threading, time
PARTS = [[0, 17835223120, "ce68d9b85e1a027f51e3a04970f3945a3fb31e1410ed32c96d1affa11508e70e"], [17835223120, 35670446240, "1d2eb75a8e7f5048098bd9239ee780aa033331f12781fcd4fb42220b0be81f14"], [35670446240, 53505669360, "11b59d7e99db6364b5d93b443b46733588a8ae46b3edda790c13bc4f9298f0a2"], [53505669360, 71340892480, "435086bfb4232226c196a10f73b0318be38dda9f6c690bd7586ee06658e886bc"], [71340892480, 89176115600, "74a4da36a4f3aa42f8382e25d7edcb77da012eec42703cc6aab0e73bc2aaf2db"], [89176115600, 107011338717, "28ca5bf8aebcd42a6760063409066ec417b11ef8a51a8fce5ae4c280a5ae58fb"]]
hits = glob.glob("/kaggle/input/**/qwen3.8-flash-next-tp1-c16-mtp5.safetensors", recursive=True)
assert len(hits) == 1, hits
PATH = hits[0]
size = os.path.getsize(PATH)
mounts = [l for l in open("/proc/mounts").read().splitlines() if "/kaggle/input" in l]
def mem():
    try:
        return int(open("/sys/fs/cgroup/memory.current").read()) / 2**30
    except Exception:
        return -1
out = {"path": PATH, "size": size, "size_ok": size == PARTS[-1][1], "mounts": mounts[:6], "mem_gib_start": mem()}
print(out, flush=True)

fd = os.open(PATH, os.O_RDONLY)
t, n, off = time.time(), 0, PARTS[2][0]  # mid-file, so pass 2 is not reading a just-read region first
while n < 8 << 30:
    b = os.pread(fd, 64 << 20, off + n)
    assert b
    n += len(b)
out["single_stream_MBps"] = round(n / 1e6 / (time.time() - t), 1)
print("single stream", out["single_stream_MBps"], "MB/s", flush=True)

res = {}
def part(i, start, end, sha):
    h, off, t = hashlib.sha256(), start, time.time()
    while off < end:
        b = os.pread(fd, min(64 << 20, end - off), off)
        assert b
        h.update(b); off += len(b)
    res[i] = {"ok": h.hexdigest() == sha, "seconds": round(time.time() - t, 1)}
    print("part", i, res[i], flush=True)
t = time.time()
th = [threading.Thread(target=part, args=(i, *p)) for i, p in enumerate(PARTS)]
[x.start() for x in th]; [x.join() for x in th]
out.update(parts=res, all_ok=out["size_ok"] and len(res) == 6 and all(r["ok"] for r in res.values()),
           parallel_MBps=round(size / 1e6 / (time.time() - t), 1), mem_gib_end=mem())
json.dump(out, open("/kaggle/working/dataset-verify.json", "w"), indent=1)
print(json.dumps(out, indent=1), flush=True)
assert out["all_ok"], out
